In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset
import torch.nn.functional as torch_F # avoid import problems becasue F is used as a variable

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import json

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *
from utils import *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [2]:
# Set global font sizes
plt.rcParams.update({
    'font.size': 18,              # Base font size
    'axes.titlesize': 20,         # Title font size
    'axes.labelsize': 15,         # Axes label font size
    'xtick.labelsize': 15,        # X-tick label font size
    'ytick.labelsize': 15,        # Y-tick label font size
    'legend.fontsize': 16,        # Legend font size
    'figure.titlesize': 18,       # Figure title size
    'figure.dpi': 300,            # Figure DPI for high resolution
    'savefig.dpi': 300            # Save figure DPI
})

# Make lines thicker
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5

# Increase marker size default
plt.rcParams['lines.markersize'] = 8

In [3]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)
biopsy_df = dfs['biopsy']

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

#notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)

eligible_patient_ids = get_valid_patient_ids(static_df=static_df, ts_data=ts_data, notes_df=notes, require_notes=False)
datapoints_limit = len(eligible_patient_ids)
#datapoints_limit = 320 * 5
dataset_splits = create_dataset_splits(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    train_size=0.8,
    test_size=0.2,
    max_patients=datapoints_limit,
    require_notes=False,
    random_state=42,
)

full_dataset = dataset_splits['full']
train_dataset = dataset_splits['train']
ts_scaler = train_dataset.ts_scaler
static_scaler = train_dataset.scaler

Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9
Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


In [4]:
def count_dataset_stats(dataset):
    """
    Count statistics from a NephroCAGEDataset
    Returns:
        - Total number of time series points across all patients
        - Notes per patient
        - Total sum of notes
    """
    # Initialize counters
    total_ts_points = 0
    notes_per_patient = {}
    total_notes = 0

    # Iterate through each patient in the dataset
    for idx in range(len(dataset)):
        sample = dataset[idx]
        patient_id = sample['patient_id']

        # Count time series points for this patient
        patient_ts_points = sample['seq_len']
        total_ts_points += patient_ts_points

        # Count notes for this patient
        if sample['notes_embeddings'] is not None:
            patient_notes_count = sample['notes_embeddings'].shape[0]
        else:
            patient_notes_count = 0

        notes_per_patient[patient_id] = patient_notes_count
        total_notes += patient_notes_count

    return {
        'total_time_series_points': total_ts_points,
        'notes_per_patient': notes_per_patient,
        'total_notes': total_notes,
        'average_ts_points_per_patient': total_ts_points / len(dataset) if len(dataset) > 0 else 0,
        'average_notes_per_patient': total_notes / len(dataset) if len(dataset) > 0 else 0
    }

# Example usage:
def print_dataset_stats(dataset):
    stats = count_dataset_stats(dataset)

    print(f"Dataset contains {len(dataset)} patients")
    print(f"Total time series points: {stats['total_time_series_points']}")
    print(f"Average time series points per patient: {stats['average_ts_points_per_patient']:.2f}")
    print(f"Total notes: {stats['total_notes']}")
    print(f"Average notes per patient: {stats['average_notes_per_patient']:.2f}")

    print("\nNotes distribution per patient:")
    for patient_id, note_count in stats['notes_per_patient'].items():
        print(f"Patient {patient_id}: {note_count} notes")

print_dataset_stats(full_dataset)

Dataset contains 3385 patients
Total time series points: 389624
Average time series points per patient: 115.10
Total notes: 366957
Average notes per patient: 108.41

Notes distribution per patient:
Patient 33: 119 notes
Patient 35: 118 notes
Patient 67: 91 notes
Patient 83: 232 notes
Patient 87: 227 notes
Patient 101: 110 notes
Patient 106: 257 notes
Patient 119: 292 notes
Patient 130: 129 notes
Patient 138: 204 notes
Patient 148: 13 notes
Patient 155: 40 notes
Patient 184: 304 notes
Patient 189: 80 notes
Patient 204: 66 notes
Patient 231: 216 notes
Patient 265: 150 notes
Patient 280: 64 notes
Patient 299: 112 notes
Patient 301: 109 notes
Patient 303: 283 notes
Patient 312: 190 notes
Patient 313: 102 notes
Patient 317: 134 notes
Patient 327: 240 notes
Patient 329: 38 notes
Patient 333: 120 notes
Patient 346: 192 notes
Patient 364: 60 notes
Patient 372: 61 notes
Patient 381: 101 notes
Patient 386: 78 notes
Patient 389: 190 notes
Patient 390: 27 notes
Patient 393: 167 notes
Patient 394: 

In [5]:
def analyze_clinical_outcomes(dataset):
    """
    Analyze clinical outcomes in the NephroCAGEDataset
    Counts:
    - Graft losses (sum and max per patient)
    - Rejections (sum and max per patient)
    - Deaths (sum and max per patient)
    """
    # Initialize counters
    total_graft_losses = 0
    total_deaths = 0
    total_rejections = 0

    # For max counts per patient
    max_rejections_per_patient = 0
    patient_with_max_rejections = None

    # Track per patient data
    rejections_per_patient = {}

    # Iterate through each patient in the dataset
    for idx in range(len(dataset)):
        sample = dataset[idx]
        patient_id = sample['patient_id']

        # Count graft losses
        if sample['graft_loss_label'].item() == 1:
            total_graft_losses += 1

        # Count deaths
        if sample['death_label'].item() == 1:
            total_deaths += 1

        # Count rejections
        rejection_days = sample['rej_rel_days']
        if rejection_days is not None:
            num_rejections = len(rejection_days)
            total_rejections += num_rejections
            rejections_per_patient[patient_id] = num_rejections

            # Update max rejections if needed
            if num_rejections > max_rejections_per_patient:
                max_rejections_per_patient = num_rejections
                patient_with_max_rejections = patient_id

    # Calculate statistics
    results = {
        'total_patients': len(dataset),
        'total_graft_losses': total_graft_losses,
        'total_deaths': total_deaths,
        'total_rejections': total_rejections,
        'max_rejections_per_patient': max_rejections_per_patient,
        'patient_with_max_rejections': patient_with_max_rejections,
        'percent_with_graft_loss': (total_graft_losses / len(dataset)) * 100 if len(dataset) > 0 else 0,
        'percent_with_death': (total_deaths / len(dataset)) * 100 if len(dataset) > 0 else 0,
        'percent_with_rejection': (len(rejections_per_patient) / len(dataset)) * 100 if len(dataset) > 0 else 0,
        'average_rejections_per_affected_patient': (total_rejections / len(rejections_per_patient)) if rejections_per_patient else 0
    }

    return results, rejections_per_patient

# Run analysis on the full_dataset
results, rejections_per_patient = analyze_clinical_outcomes(full_dataset)

# Print summary statistics
print(f"Dataset Statistics for {results['total_patients']} patients:")
print(f"-------------------------------------------")
print(f"Graft Losses: {results['total_graft_losses']} ({results['percent_with_graft_loss']:.1f}% of patients)")
print(f"Deaths: {results['total_deaths']} ({results['percent_with_death']:.1f}% of patients)")
print(f"Rejections: {results['total_rejections']} total across {len(rejections_per_patient)} patients")
print(f"          ({results['percent_with_rejection']:.1f}% of patients experienced rejection)")
print(f"Maximum rejections for a single patient: {results['max_rejections_per_patient']} (Patient ID: {results['patient_with_max_rejections']})")
print(f"Average rejections per affected patient: {results['average_rejections_per_affected_patient']:.1f}")

# Print patients with most rejections (top 5)
print("\nTop 5 patients with most rejections:")
top_rejection_patients = sorted(rejections_per_patient.items(), key=lambda x: x[1], reverse=True)[:5]
for patient_id, count in top_rejection_patients:
    print(f"Patient {patient_id}: {count} rejections")

Dataset Statistics for 3385 patients:
-------------------------------------------
Graft Losses: 587 (17.3% of patients)
Deaths: 1040 (30.7% of patients)
Rejections: 387 total across 267 patients
          (7.9% of patients experienced rejection)
Maximum rejections for a single patient: 6 (Patient ID: 8055)
Average rejections per affected patient: 1.4

Top 5 patients with most rejections:
Patient 8055: 6 rejections
Patient 8864: 5 rejections
Patient 1871: 4 rejections
Patient 6179: 4 rejections
Patient 7212: 4 rejections


In [6]:
models = [
    'Vanilla LSTM',
    'Time-Aware LSTM',
    'Time-Aware LSTM with Attention'
]

errors = [0.72, 0.54, 0.31]
std_devs = [0.02, 0.01, 0.01]

print("Model Reconstruction Error Comparison during Training:")
print("--------------------------------------")
print(f"{'Model':<30} {'Reconstruction Error':<20}")
print("--------------------------------------")
for model, err, std in zip(models, errors, std_devs):
    print(f"{model:<30} {err:.2f} (±{std:.2f})")

Model Reconstruction Error Comparison during Training:
--------------------------------------
Model                          Reconstruction Error
--------------------------------------
Vanilla LSTM                   0.72 (±0.02)
Time-Aware LSTM                0.54 (±0.01)
Time-Aware LSTM with Attention 0.31 (±0.01)


In [ ]:
# Load the JSON data from the specified path
with open('../data/results/horizons.json', 'r') as f:
    results = json.load(f)

# Create the visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
categories = ["graft_loss", "graft_rejection", "mortality"]
period_labels = {"30_day": "30-day", "90_day": "90-day", "180_day": "180-day", "360_day": "360-day"}
colors = {"30_day": "#1f77b4", "90_day": "#ff7f0e", "180_day": "#2ca02c", "360_day": "#d62728"}
markers = {"30_day": "o", "90_day": "s", "180_day": "^", "360_day": "d"}
line_styles = {"30_day": "-", "90_day": "--", "180_day": "-.", "360_day": ":"}

x = list(range(1, 26))  # Horizons from 1 to 25

for i, category in enumerate(categories):
    ax = axes[i]
    for period, values in results[category].items():
        ax.plot(x, values, label=period_labels[period],
                color=colors[period], marker=markers[period],
                linestyle=line_styles[period], markersize=5)

    ax.set_xlabel('Prediction Horizon')
    if i == 0:
        ax.set_ylabel('AUC-ROC')
    ax.set_title(f'{category.replace("_", " ").title()}')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_xticks(list(range(1, 26, 2)))
    ax.set_xlim(0.5, 25.5)
    ax.set_ylim(0.55, 0.9)

# Add a common legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))

#plt.tight_layout()
plt.subplots_adjust(bottom=0.15)  # Adjust for the legend
#plt.suptitle('AUC-ROC Across Different Prediction Horizons, Only TS data', fontsize=16)

# Save the figure
plt.savefig('../plots/auc_roc_prediction_horizons.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
with open('../data/results/final_res.json', 'r') as f:
    data_sources = json.load(f)

# Define colors and markers
source_colors = {
    'time_series_only': '#1f77b4',
    'time_series_static': '#ff7f0e',
    'time_series_static_notes': '#2ca02c'
}
source_markers = {
    'time_series_only': 'o',
    'time_series_static': 's',
    'time_series_static_notes': '^'
}
source_labels = {
    'time_series_only': 'Time Series Only',
    'time_series_static': 'Time Series + Static',
    'time_series_static_notes': 'Time Series + Static + Notes'
}
period_labels = {
    '30_day': '30-day',
    '90_day': '90-day',
    '180_day': '180-day',
    '360_day': '360-day'
}

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
#fig.suptitle('Comparison of Data Sources on Prediction Tasks', fontsize=16)

# Tasks to plot
tasks = ['graft_loss', 'graft_rejection', 'mortality']
task_display_names = {
    'graft_loss': 'Graft Loss',
    'graft_rejection': 'Graft Rejection',
    'mortality': 'Mortality'
}

# Find global min and max values across all tasks
global_min = 1.0
global_max = 0.0

for task in tasks:
    for source in ['time_series_only', 'time_series_static', 'time_series_static_notes']:
        means = [item['mean'] for item in data_sources[task][source]]
        stds = [item['std'] for item in data_sources[task][source]]

        min_with_error = min([m - s for m, s in zip(means, stds)])
        max_with_error = max([m + s for m, s in zip(means, stds)])

        global_min = min(global_min, min_with_error)
        global_max = max(global_max, max_with_error)

# Add padding to the y-axis limits
y_range = global_max - global_min
padding = y_range * 0.07
y_min = max(0.5, global_min - padding)
y_max = min(1.0, global_max + padding)

# For each task
for i, task in enumerate(tasks):
    ax = axes[i]
    ax.set_title(task_display_names[task])

    # Set the global y-axis limits
    ax.set_ylim(y_min, y_max)

    # Position for each time period
    x_pos = np.arange(4)

    # For each time period
    for p_idx, period in enumerate(['30_day', '90_day', '180_day', '360_day']):
        # Draw vertical reference line
        ax.axvline(x=p_idx, color='gray', linestyle='-', alpha=0.2, zorder=0)

        # For each data source
        for s_idx, source in enumerate(['time_series_only', 'time_series_static', 'time_series_static_notes']):
            # Find the data point
            source_data = data_sources[task][source]
            for item in source_data:
                if item['period'] == period:
                    mean = item['mean']
                    std = item['std']

                    # Plot marker
                    ax.errorbar(p_idx, mean, yerr=std,
                               fmt=source_markers[source], markersize=10,
                               label=source_labels[source] if (i == 0 and p_idx == 0) else "",
                               color=source_colors[source], capsize=4, elinewidth=2, zorder=2)

    # Add labels for each time period
    ax.set_xticks(x_pos)
    ax.set_xticklabels([period_labels[p] for p in ['30_day', '90_day', '180_day', '360_day']])

    if i == 0:
        ax.set_ylabel('AUC-ROC')

    ax.grid(True, linestyle='--', alpha=0.7, axis='y')

# Add a common legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))

# Layout adjustments
#plt.tight_layout()
#plt.subplots_adjust(bottom=0.15, top=0.9)

# Save the figure
plt.savefig('../plots/data_sources_comparison.jpg', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
# Load the model comparison data from JSON file
with open('../data/results/lms_comp.json', 'r') as f:
    model_data = json.load(f)

# Define time periods and their display labels
periods = ["30_day", "90_day", "180_day", "360_day"]
period_labels = ["30-day", "90-day", "180-day", "360-day"]

# Define tasks and their display names
tasks = ["graft_loss", "graft_rejection", "mortality"]
task_display_names = {
    "graft_loss": "Graft Loss",
    "graft_rejection": "Graft Rejection",
    "mortality": "Mortality"
}

# Define models and their display names with distinct colors and markers
# Removed "med-gte-hybrid" from the list
models = ["gte-large", "med-gte-hybrid-DE"]
model_display_names = {
    "gte-large": "gte-large",
    "med-gte-hybrid-DE": "med-gte-hybrid-de"
}

# Updated colors (kept the same colors for consistency)
model_colors = {
    "gte-large": "blue",
    "med-gte-hybrid-DE": "green"
}

# Updated markers
model_markers = {
    "gte-large": "D",
    "med-gte-hybrid-DE": "o"
}

# Create a figure with subplots for each task
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
#fig.suptitle('Model Performance Comparison Across Tasks')

# For each task
for i, task in enumerate(tasks):
    ax = axes[i]

    # Position for each time period
    x_pos = np.arange(4)

    # For each time period
    for p_idx, period in enumerate(periods):
        # Draw vertical reference line
        ax.axvline(x=p_idx, color='gray', linestyle='-', alpha=0.2, zorder=0)

        # For each model (now without med-gte-hybrid)
        for j, model in enumerate(models):
            # Extract data for this model, task, and period
            mean = model_data[task][model][period]["mean"]
            std = model_data[task][model][period]["std"]

            # Plot marker with distinct markers
            ax.errorbar(p_idx, mean, yerr=std,
                       fmt=model_markers[model], markersize=10,
                       label=model_display_names[model] if (i == 0 and p_idx == 0) else "",
                       color=model_colors[model], capsize=4, elinewidth=2, zorder=2)

    # Add labels for each time period
    ax.set_xticks(x_pos)
    ax.set_xticklabels(period_labels)

    # Set title and y-axis label
    ax.set_title(task_display_names[task])
    if i == 0:
        ax.set_ylabel('AUC-ROC')

    # Set y-axis limits
    ax.set_ylim(0.7, 1.0)

    # Add grid for better readability
    ax.grid(True, linestyle='--', alpha=0.7, axis='y')

# Add a common legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05))

# Layout adjustments
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)  # Make room for the legend

# Save the figure
plt.savefig('../plots/model_performance_comparison.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
models = [
    'DFKI CDSS Paper',
    'Time-Aware LSTM with Notes'
]

# Graft loss metrics
graft_loss_auc = [0.9415, 0.9648]
graft_loss_sen = [0.6667, 0.7500]
graft_loss_spe = [0.9247, 0.9512]

# Graft rejection metrics
graft_rejection_auc = [0.7465, 0.8471]
graft_rejection_sen = [0.5600, 0.6927]
graft_rejection_spe = [0.6947, 0.8618]

print("90-Day Graft Loss Prediction Performance:")
print("----------------------------------------------")
print(f"{'Model':<35} {'AUC':<10} {'Sensitivity':<15} {'Specificity':<15}")
print("----------------------------------------------")
for model, auc, sen, spe in zip(models, graft_loss_auc, graft_loss_sen, graft_loss_spe):
    print(f"{model:<35} {auc:.4f}    {sen:.4f}         {spe:.4f}")

print("\n90-Day Graft Rejection Prediction Performance:")
print("----------------------------------------------")
print(f"{'Model':<35} {'AUC':<10} {'Sensitivity':<15} {'Specificity':<15}")
print("----------------------------------------------")
for model, auc, sen, spe in zip(models, graft_rejection_auc, graft_rejection_sen, graft_rejection_spe):
    print(f"{model:<35} {auc:.4f}    {sen:.4f}         {spe:.4f}")

In [ ]:
# Load the combined JSON file
with open('../data/results/shap.json', 'r') as file:
    data = json.load(file)

# Create output directory if it doesn't exist
os.makedirs('plots', exist_ok=True)

# Tasks to plot (excluding 'overall')
tasks_to_plot = ['graft_loss', 'graft_rejection', 'mortality']

# Create a bar chart for each task
for task_name in tasks_to_plot:
    if task_name in data['prediction_tasks']:
        # Get data for this task
        task_data = data['prediction_tasks'][task_name]

        # Create DataFrame
        df = pd.DataFrame(task_data)

        # Sort by importance in descending order
        df = df.sort_values('importance', ascending=True)

        # Create the figure
        plt.figure(figsize=(12, 10))

        # Plot horizontal bars
        plt.barh(df['feature'], df['importance'], color='#1f77b4')

        # Add labels and title
        plt.xlabel('Mean |SHAP value|', fontsize=12)
        title = f'Feature Importance for {task_name.replace("_", " ").title()} Prediction'
        plt.title(title, fontsize=14)

        # Adjust y-axis labels
        plt.yticks(fontsize=10)

        # Remove top and right spines
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)

        # Adjust layout
        plt.tight_layout()

        # Save the figure
        plt.savefig(f'../plots/feature_importance_{task_name}.png', dpi=300, bbox_inches='tight')

        # Display the plot in the notebook
        plt.show()

print("All plots created successfully!")

In [ ]:
# Create a single figure with three subplots side by side
plt.figure(figsize=(20, 10))

# Tasks to plot
tasks_to_plot = ['graft_loss', 'graft_rejection', 'mortality']
titles = [task.replace("_", " ").title() for task in tasks_to_plot]

# Process each task in a subplot
for i, task_name in enumerate(tasks_to_plot):
    # Create subplot
    ax = plt.subplot(1, 3, i+1)

    if task_name in data['prediction_tasks']:
        # Get data for this task
        task_data = data['prediction_tasks'][task_name]

        # Create DataFrame
        df = pd.DataFrame(task_data)

        # Sort by importance in descending order and get top 10
        df = df.sort_values('importance', ascending=False).head(10)
        # Reverse order for horizontal bar chart (to have highest at top)
        df = df.sort_values('importance', ascending=True)

        # Plot horizontal bars
        ax.barh(df['feature'], df['importance'], color='#1f77b4')

        # Add labels and title
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title(titles[i])

        # Remove top and right spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Adjust y-axis labels
        ax.tick_params(axis='y')

# Adjust layout
plt.tight_layout()

# Save the figure
plt.savefig('../plots/combined_top10_feature_importance.jpg', dpi=300, bbox_inches='tight')

# Display the plot
plt.show()

print("Combined plot created successfully!")

In [ ]:
models = [
    'Best model',
    'Random shuffling of timesteps'
]
errors = [0.31, 0.96]
auc_graft_loss = [0.96, 0.71]
auc_graft_rejection = [0.85, 0.63]

print("Model Performance Comparison when Shuffling Timesteps:")
print("-----------------------------------------------------------------------------------------------")
print(f"{'Model':<30} {'Reconstruction Error':<20} {'AUC 90d Graft Loss':<20} {'AUC 90d Rejection':<20}")
print("------------------------------------------------------------------------------------------------")
for model, err, std, auc_loss, auc_rej in zip(models, errors, std_devs, auc_graft_loss, auc_graft_rejection):
    print(f"{model:<30} {err:.2f} {'':20} {auc_loss:.2f}{'':<12} {auc_rej:.2f}")

In [ ]:
def plot_saved_attention_weights(json_path='../data/results/temporal_attention.json', max_days=None, label_interval=50):
    """
    Read the saved attention weights JSON and plot them with proper day labels on x-axis.

    Args:
        json_path: Path to the JSON file containing attention weights
        max_days: Maximum number of days to show on x-axis (None for all days)
        label_interval: Show x-axis labels every N days
    """
    # Check if file exists
    if not os.path.exists(json_path):
        print(f"Error: File {json_path} not found!")
        return

    # Load the JSON data
    with open(json_path, 'r') as f:
        attention_data = json.load(f)

    # Plot each sample
    for sample_key, sample_data in attention_data.items():
        patient_id = sample_data["patient_id"]
        attn_weights = sample_data["attention_weights"]
        time_labels = sample_data["time_labels"]

        # Convert time labels to day numbers
        # Assuming time_labels are in days already; adjust if they're in hours or another unit
        days = [int(t) for t in time_labels]

        # Find min and max days
        min_day = min(days)
        max_day = max(days)

        # Limit to max_days if specified
        if max_days is not None:
            max_day = min(max_day, min_day + max_days - 1)

        # Create a continuous range of days
        all_days = list(range(min_day, max_day + 1))

        # Initialize attention weights for all days (0 for days without measurements)
        day_to_attention = {day: 0 for day in all_days}

        # Fill in actual attention weights
        for day, weight in zip(days, attn_weights):
            if day <= max_day:  # Skip days beyond max_day
                day_to_attention[day] = weight

        # Convert dictionary to lists for plotting
        x_values = list(day_to_attention.keys())
        y_values = list(day_to_attention.values())

        # Create bar plot
        plt.figure(figsize=(12, 6))
        bars = plt.bar(x_values, y_values, color='royalblue')

        # Highlight days with actual measurements with a different color
        for i, day in enumerate(x_values):
            if day in days and day <= max_day:
                bars[i].set_color('royalblue')
            else:
                bars[i].set_color('lightgrey')  # Days without measurements

        # Set up x-axis ticks with reduced density
        xticks = [day for day in x_values if day % label_interval == 0]
        plt.xticks(xticks, xticks)

        # Set axis limits
        plt.xlim(min_day - 0.5, max_day + 0.5)

        # Add titles and labels
        #plt.title(f'Temporal Attention to Timesteps - Patient {patient_id}')
        plt.xlabel('Days')
        plt.ylabel('Attention Weight')
        plt.grid(axis='y', linestyle='--', alpha=0.3)

        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='royalblue', label='Attention weight for observed data points'),
            #Patch(facecolor='lightgrey', label='Days without measurements')
        ]
        plt.legend(handles=legend_elements, loc='upper right')

        plt.tight_layout()
        plt.savefig("../plots/temp_att.jpg")
        plt.show()

# Plot the saved attention weights with appropriate parameters
plot_saved_attention_weights(max_days=720, label_interval=50)

In [ ]:
## DCI Metrics

# - Disentanglement: Each latent dimension captures at most one generative factor
# - Completeness: Each generative factor is captured by at most one latent dimension
# - Informativeness: Amount of information the latent space contains about factors

models = [
    'Baseline model (only reconstruction error)',
    '+ Orthogonality penalty',
    '+ Feature disentanglement penalty'
]
disentanglement = [0.28, 0.61, 0.87]
completeness = [0.42, 0.76, 0.78]
informativeness = [0.78, 0.83, 0.85]

df = pd.DataFrame({
    'Model': models,
    'Disentanglement': disentanglement,
    'Completeness': completeness,
    'Informativeness': informativeness
})

print("DCI Metrics Comparison (higher is better for all metrics):")
print("----------------------------------------------------------")
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))